<div class = "alert-block alert-info">
<font size="5"><b>Traitement ASTER</b></font>
</div>

<code style="background:gray;color:white">
<b>Author:</b> Virginie Masson
<strong><em>Date:</em></strong> <em>7 October 2024</em></code>
</p>
<em>This notebook was prepared on JupyterLab. If JupyterLab is not used to read this notebook, it is possible that some functionality/display does not work.</em>

# <span style="color:magenta;">ASTER Data Processing for Geological Projects</span>
<p>
This notebook aims to guide you through the process of ASTER (Advanced Spaceborne Thermal Emission and Reflection Radiometer) data processing, specifically for geological projects. You will find detailed explanations and Python code examples to help you conduct your geospatial analysis.
</p>

---

<div class="alert-block" style="background-color:#ff66cc; padding:10px; border-radius:5px;">
<font size="5" color="black"><b>Introduction</b></font>
</div>

<p>
ASTER (Advanced Spaceborne Thermal Emission and Reflection Radiometer) imagery is a multispectral sensor onboard NASA's Terra satellite. It captures images of the Earth's surface in 14 spectral bands ranging from infrared to thermal infrared. ASTER is particularly useful in geology for studying minerals, rocks, soil types, and geological structures.
<b>This process is run on Python using the rasterio, NumPy, and matplotlib.pyplot.</b>
</p>

<p> The ASTER (Advanced Spaceborne Thermal Emission and Reflection Radiometer) GED C2 (Global Emissivity Dataset Version 2) Subset is a data product derived from the ASTER instrument onboard NASA's Terra satellite. It is a high-resolution global dataset on the surface emissivity of the Earth, covering non-polar regions. </p>
<br>
<p>Here are some key points about ASTER product:</p>
<ol> 
<li> <code style="background:gray;color:white">Thermal emissivity:</code> Emissivity is a measure of a surface's ability to emit energy as thermal radiation. The ASTER GED C2 product provides detailed data on global emissivity with a <b> 100-meter resolution </b>, making it a valuable tool for analyzing the thermal properties of materials on Earth's surface. </li>
<br>
<li><code style="background:gray;color:white">Spectral bands:</code></li>
    <ol>
    <li><b>VNIR Bands (Visible and Near Infrared)</b>
        <ol>
            <li>Band 1 (0.45-0.52 µm): Blue</li>
            <li>Band 2 (0.52-0.60 µm): Green</li>
            <li>Band 3 (0.63-0.69 µm): Red</li>
            <li>Band 4 (0.76-0.86 µm): Near Infrared (NIR)</li>
            <li>Band 5 (0.85-0.88 µm): Near Infrared (NIR)</li>
        </ol>
        <b><em>These bands are used for vegetation detection, water quality analysis, and soil mapping.</em></b>
    </li>
    <li><b>SWIR Bands (Shortwave Infrared)</b>
        <ol>
            <li>Band 6 (1.60-1.70 µm): </li>
            <li>Band 7 (2.10-2.30 µm): </li>
            <li>Band 8 (2.33-2.36 µm): </li>
            <li>Band 9 (2.36-2.39 µm): </li>
            <li>Band 10 (2.40-2.50 µm): </li>
        </ol>
        <b><em>These bands are particularly useful for identifying minerals, detecting soil moisture, and conducting geological analyses.</em></b>
    </li>
    <li><b>TIR Bands (Thermal Infrared)</b>
        <ol>
            <li>Band 11 (8.125-8.475 µm): </li>
            <li>Band 12 (8.475-8.825 µm): </li>
            <li>Band 13 (10.25-10.95 µm): </li>
            <li>Band 14 (10.95-11.65 µm): </li>
        </ol>
        <b><em>These bands measure the thermal radiation emitted by the surface, allowing for the analysis of surface temperatures and the evaluation of the emissivity of materials.</em></b>
    </li></ol>
<li><code style="background:gray;color:white">Version 2:</code> Version 2 of this dataset brings improvements over the previous version, including better correction for atmospheric effects and errors related to observation angles, leading to more accurate data.</li>        
<li><code style="background:gray;color:white">Subset: </code> A 'subset' is a specific extraction or subset of the global dataset. These subsets allow users to download only the data covering a specific area of interest, reducing the amount of data to process and optimizing analysis time.</li>
</ol>

<br>
<div class = "alert-block alert-warning">
<p> <em>
It is important to note that for this project, the <b>ASTER GED C2 Subset will be used.</b>
</em> </p>
<p>These data are available for users wanting to generate custom Level-2 products using the Collection 2 surface reflectance and surface temperature algorithms. 
<b>Emissivity data from Advanced Spaceborne Thermal Emission and Reflection Radiometer (ASTER) Global Emissivity Dataset (GED) bands 13, 14, and Normalized Difference Vegetation Index (NDVI) are used in the ST algorithm for Landsat 4-9 data.</b>
The ASTER GED is divided into a separate file for each 1×1-degree area of all major non-frozen land masses of the Earth’s land surface. Antarctica and some small islands are not covered. The USGS extracts Emissivity, NDVI, and geolocation information from each ASTER GED file and stores them in a new Hierarchical Data Format Version 5 (HDF5) file. 
<br> 
<b>Within a file, the Emissivity and NDVI data are recorded in 100m grids. The Emissivity “Mean” and “SDev” datasets contain a value for ASTER bands 10, 11, 12, 13, and 14, although only bands 13 and 14 are used in the ST algorithm. The latitude and longitude values at each grid point are also stored. The new files are in the same format and spatial resolution as the original files.</b></p>
</div> 

### **Steps for Processing:**

1. **Loading ASTER Emissivity Data:**
   You have two files: `AG100.v003.43.063.0001.subset` and `AG100.v003.43.062.0001.subset`. These files contain emissivity data in the thermal infrared (TIR), used to identify geological materials.

2. **Data Preprocessing:**
   - Load the raster files.
   - Create a mosaic if necessary (if the files cover adjacent areas).

3. **Identifying Geological Sets:**
   - Use supervised or unsupervised classification to group emissivity signatures corresponding to different types of materials.
   - Apply spectral signature analysis.

4. **Creating the Export Raster for Visualization in QGIS:**
   - Generate a raster representing the identified geological sets.
   - Export the result as a GeoTIFF for import into QGIS.

<div class="alert-block" style="background-color:#ff66cc; padding:10px; border-radius:5px;">
<font size="5" color="black"><b>ASTER PROCESSING</b></font>
</div>

In [1]:
import os
import h5py
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

1. **`rasterio`:**
   - **Purpose:** Used for reading and writing raster data (e.g., GeoTIFF files). It handles various raster file formats and provides functions to read metadata, manipulate raster data, and export results.
   - **Usage in Script:** Loads the ASTER emissivity data and saves the classified output raster.

2. **`numpy`:**
   - **Purpose:** A fundamental library for numerical computing in Python. It provides support for arrays, matrices, and a wide range of mathematical functions.
   - **Usage in Script:** Handles the manipulation of raster data (e.g., reshaping arrays) and performs numerical operations necessary for processing.

3. **`matplotlib.pyplot`:**
   - **Purpose:** A plotting library used for creating static, interactive, and animated visualizations in Python. It provides a MATLAB-like interface for plotting.
   - **Usage in Script:** Displays images of the raster data and the results of the geological classification.

4. **`sklearn.cluster`:**
   - **Purpose:** Part of the scikit-learn library, this module provides tools for machine learning, specifically for clustering and classification algorithms. KMeans is used for grouping data points based on their features.
   - **Usage in Script:** Performs unsupervised classification of the emissivity data to identify geological sets based on their spectral signatures.

In [4]:
# Chemin vers le fichier HDF5
file_path1 = r"D:\PROJET\UZB_North_Bukantau\04_GIS\02_RASTER\05_TELEDETECTION\ASTER\AG100.v003.43.062.0001.subset"
file_path2 = r"D:\PROJET\UZB_North_Bukantau\04_GIS\02_RASTER\05_TELEDETECTION\ASTER\AG100.v003.43.063.0001.subset"

In [5]:
# Explorer le premier fichier HDF5
print("Exploring file 1:")
with h5py.File(file_path1, 'r') as f:
    # Afficher les clés principales (bandes ou groupes)
    print(list(f.keys()))

# Explorer le deuxième fichier HDF5
print("\nExploring file 2:")
with h5py.File(file_path2, 'r') as f:
    # Afficher les clés principales (bandes ou groupes)
    print(list(f.keys()))

Exploring file 1:


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'D:\PROJET\UZB_North_Bukantau\04_GIS\02_RASTER\05_TELEDETECTION\ASTER\AG100.v003.43.062.0001.subset', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [6]:

file_path1 = r"D:\PROJET\UZB_North_Bukantau\04_GIS\02_RASTER\05_TELEDETECTION\ASTER\AG100.v003.43.062.0001.subset"

# Vérifiez si le fichier existe
if os.path.exists(file_path1):
    with h5py.File(file_path1, 'r') as f:
        print("File opened successfully.")
        print(list(f.keys()))  # Affiche les sous-datasets
else:
    print(f"File does not exist: {file_path1}")

File does not exist: D:\PROJET\UZB_North_Bukantau\04_GIS\02_RASTER\05_TELEDETECTION\ASTER\AG100.v003.43.062.0001.subset
